# BƯỚC 3 · T-A: PHÂN LOẠI GIỌNG THẬT / GIẢ TRÊN VIVOS

Notebook tự dựng thí nghiệm **riêng** với dữ liệu VIVOS, không đưa 6.285 clip Bước 2 vào T-A. Dự kiến 25 người × 15 câu = **375 clip thật + 1.125 clip giả**, tạo bởi VieNeu, OmniVoice và OmDiOm-ft; 18 người train / 7 người test đối với **bộ phân loại**.

**Trước khi chạy:** Add Input VIVOS Vietnamese (`thinh127/vivos-vietnamese`) và notebook `Train_CheckPoint` chứa `omdiom_full/checkpoint-4000`; chọn **GPU T4**, Internet ON, rồi **Save Version → Save & Run All**. Không cần 7 `.pt` và không sửa code. Kaggle không đọc ổ D:.

Mỗi người có một bản ghi tham chiếu *khác* các câu đem đánh giá. Bản thật/giả của cùng cặp ở cùng tập. Cùng chữ và người giúp kiểm soát, nhưng không loại bỏ mọi nhiễu codec, ghi âm hoặc nội dung. Audio được lưu thống nhất 16 kHz FLAC. Output mới chỉ cho thấy test là 7 người không nằm trong train **của bộ phân loại**; không biết liệu các bộ TTS nền đã gặp VIVOS khi huấn luyện hay chưa.

Sinh 1.125 clip thực tế có thể lâu; thời gian phụ thuộc độ dài câu và GPU. Lưu tiến độ ở output. Các ngưỡng thời lượng/âm lượng chỉ sàng lọc ban đầu, **không chứng minh đọc đủ câu**: bắt buộc nghe kiểm tra. Nếu một cặp chưa đủ 3 bộ sinh, cặp đó không đưa vào train/test; báo cáo ghi rõ và vẫn đóng gói phần đã sinh.

In [ ]:
# Ô 1 · MÔI TRƯỜNG VÀ THIẾT LẬP
import os,sys,time,glob,json,gzip,shutil,subprocess,hashlib,random,re,inspect,importlib,importlib.util
from pathlib import Path
for k in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS","NUMEXPR_NUM_THREADS"):os.environ[k]="1"
VIVOS=Path("/kaggle/input/datasets/thinh127/vivos-vietnamese/vivos/train")
RA=Path("/kaggle/working/ta_vivos");RA.mkdir(parents=True,exist_ok=True)
SEED=20261103;NUM_STEP=32;SR=16000;SO_NGUOI=25;SO_CAU=15;SO_TEST=7;LAN_THU=4;GIO_TOI_DA=9.0
BO=["vieneu","omnivoice","omdiom"];N_WORK=2

def can(pkg,mod):
    if importlib.util.find_spec(mod) is not None:print("✓",pkg);return
    print("Cài",pkg,"— cần Internet")
    p=subprocess.run([sys.executable,"-m","pip","install","-q",pkg],capture_output=True,text=True)
    importlib.invalidate_caches()
    if p.returncode!=0 or importlib.util.find_spec(mod) is None:raise RuntimeError("Không cài được "+pkg+": "+(p.stderr or p.stdout)[-1400:])
for pkg,mod in [("numpy","numpy"),("pandas","pandas"),("scipy","scipy"),("soundfile","soundfile"),("scikit-learn","sklearn"),("matplotlib","matplotlib"),("joblib","joblib"),("vieneu","vieneu"),("omnivoice","omnivoice")]:can(pkg,mod)
import numpy as np,pandas as pd,soundfile as sf,torch,scipy,sklearn,joblib,matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.signal import resample_poly,stft
from joblib import Parallel,delayed
if not VIVOS.is_dir():raise FileNotFoundError("Không thấy VIVOS tại "+str(VIVOS)+". Add Input dataset thinh127/vivos-vietnamese.")
if not torch.cuda.is_available():raise RuntimeError("Cần GPU CUDA: bật T4 ở Settings trước khi chạy.")
print("✓ VIVOS:",VIVOS,"· GPU:",torch.cuda.get_device_name(0),"· NUM_STEP:",NUM_STEP)


In [ ]:
# Ô 2 · ĐỌC VIVOS THỰC TẾ: prompts.txt CÓ DẠNG 'MÃ CÂU + VĂN BẢN'
# Cũng hỗ trợ định dạng ngăn cách bằng |; không phụ thuộc tên prompts-train.txt.
files=sorted(set(VIVOS.rglob("prompts*.txt")) | set(VIVOS.rglob("prompts*.txt.gz")))
if not files:raise FileNotFoundError("Không tìm thấy prompts*.txt trong "+str(VIVOS))
texts={}
for p in files:
    op=gzip.open if p.suffix==".gz" else open
    with op(p,"rt",encoding="utf-8-sig",errors="replace") as f:
        for line in f:
            line=line.strip()
            if not line:continue
            if "|" in line: pieces=line.split("|",1)
            else: pieces=line.split(maxsplit=1)
            if len(pieces)!=2:continue
            key=Path(pieces[0].replace("\\","/")).stem
            text=pieces[1].strip()
            if not text:continue
            if key in texts and texts[key]!=text:raise ValueError("Mã câu có hai transcript khác nhau: "+key)
            texts[key]=text
wavs=sorted(VIVOS.rglob("*.wav"));lookup={}
for p in wavs:
    key=p.stem
    if key in lookup:raise ValueError("Mã WAV không duy nhất: "+key)
    lookup[key]=p
print(f"✓ prompts: {len(files)} tệp · {len(texts)} transcript · {len(wavs)} WAV")
if not texts or not wavs:raise RuntimeError("Không đọc được transcript hoặc WAV VIVOS.")
from collections import defaultdict
by_spk=defaultdict(list);loi_info=0
for key in sorted(set(texts)&set(lookup)):
    p=lookup[key]
    # VIVOS chuẩn waves/VIVOSSPKxx/VIVOSSPKxx_Rxxx.wav.
    spk=key.split("_")[0]
    if not re.fullmatch(r"VIVOSSPK\d+",spk,re.IGNORECASE):continue
    try:
        info=sf.info(p)
        dur=info.frames/info.samplerate
        if info.samplerate<=0 or not np.isfinite(dur):continue
    except Exception:
        loi_info+=1;continue
    by_spk[spk].append({"ma":key,"text":texts[key],"path":str(p),"giay":dur})
# Chọn tham chiếu nguyên câu 3–8s. Không cắt audio mà vẫn giữ transcript cũ.
def norm(t):return re.sub(r"\s+"," ",t.strip().casefold())
eligible={}
for spk,rows in sorted(by_spk.items()):
    refs=[r for r in rows if 3.0<=r["giay"]<=8.0 and len(r["text"].split())>=3]
    targets=[r for r in rows if 2.0<=r["giay"]<=10.0 and 3<=len(r["text"].split())<=30]
    if not refs:continue
    rr=random.Random(int(hashlib.sha256((str(SEED)+spk).encode()).hexdigest()[:12],16))
    rr.shuffle(refs);rr.shuffle(targets)
    for ref in refs:
        # Tránh cùng file và cùng transcript với clip tham chiếu.
        items=[r for r in targets if r["ma"]!=ref["ma"] and norm(r["text"])!=norm(ref["text"])]
        if len(items)>=SO_CAU:
            eligible[spk]=(ref,items[:SO_CAU]);break
print(f"✓ {len(by_spk)} người trong VIVOS train · {len(eligible)} đủ tham chiếu 3–8s và 15 câu · lỗi metadata {loi_info}")
if len(eligible)<SO_NGUOI:raise RuntimeError(f"Chỉ {len(eligible)} người đủ điều kiện; cần {SO_NGUOI}. Không hạ số người âm thầm.")
rng=random.Random(SEED);speakers=sorted(eligible);rng.shuffle(speakers);speakers=speakers[:SO_NGUOI]
test_people=set(speakers[:SO_TEST]);train_people=set(speakers[SO_TEST:]);design=[]
for spk in sorted(speakers):
    ref,items=eligible[spk]
    for r in items:
        design.append(dict(pair_id=spk+"__"+r["ma"],speaker=spk,tap="test" if spk in test_people else "train",cau_id=r["ma"],text=r["text"],wav_that=r["path"],giay_that=r["giay"],ref_id=ref["ma"],ref_text=ref["text"],ref_wav=ref["path"]))
design=pd.DataFrame(design).sort_values(["tap","speaker","cau_id"]).reset_index(drop=True)
if len(design)!=SO_NGUOI*SO_CAU or design.pair_id.duplicated().any():raise ValueError("Thiết kế không đủ số cặp duy nhất")
if set(design[design.tap=="train"].speaker)&set(design[design.tap=="test"].speaker):raise RuntimeError("Rò rỉ người nói")
shared=set(map(norm,design[design.tap=="train"].text))&set(map(norm,design[design.tap=="test"].text))
print(f"✓ {len(train_people)} người train · {len(test_people)} người test · {len(design)} cặp · {len(shared)} nội dung text trùng chính xác giữa train/test")
print("  Lưu ý: kiểm tra text trùng chính xác không loại bỏ mọi mẫu câu tương tự.")
# Vân tay thiết kế cho resume; thông tin lựa chọn và transcript được ghi rõ.
key_cols=["pair_id","speaker","tap","cau_id","text","ref_id","ref_text"]
design_sig=hashlib.sha256(design[key_cols].to_csv(index=False).encode("utf-8")).hexdigest()


In [ ]:
# Ô 3 · KHÔI PHỤC TIẾN ĐỘ CÓ CÙNG THIẾT KẾ (NẾU ĐƯỢC ADD INPUT)
existing=RA/"thiet_ke.json"
if existing.is_file():
    old=json.loads(existing.read_text(encoding="utf-8"))
    if old.get("sha256")!=design_sig:raise RuntimeError("Working đã có thiết kế T-A khác; dừng để không trộn dữ liệu.")
else:
    # Có thể Add Input output T-A cũ vào phiên mới; chỉ khôi phục bản khớp fingerprint.
    found=[]
    for f in glob.glob("/kaggle/input/**/ta_vivos/thiet_ke.json",recursive=True):
        try:
            info=json.loads(Path(f).read_text(encoding="utf-8"));parent=Path(f).parent
            if info.get("sha256")==design_sig:
                n=len(list((parent/"audio").rglob("*.flac"))) if (parent/"audio").is_dir() else 0
                found.append((n,str(parent)))
        except Exception as e:print("Không đọc được dấu phục hồi:",f,e)
    if found:
        found.sort();chosen=Path(found[-1][1]);print("Khôi phục từ",chosen,"·",found[-1][0],"clip")
        for folder in ("audio","ref"):
            if (chosen/folder).is_dir():shutil.copytree(chosen/folder,RA/folder,dirs_exist_ok=True)
        if (chosen/"nhat_ky_sinh.json").is_file():shutil.copy2(chosen/"nhat_ky_sinh.json",RA/"nhat_ky_sinh.json")
    existing.write_text(json.dumps({"sha256":design_sig,"seed":SEED,"vivos":str(VIVOS)},ensure_ascii=False,indent=2),encoding="utf-8")
design.to_csv(RA/"thiet_ke_cap.csv",index=False,encoding="utf-8-sig")
# Danh sách tham chiếu riêng; không dùng clip mục tiêu làm tham chiếu.
refs=design[["speaker","tap","ref_id","ref_text","ref_wav"]].drop_duplicates()
refs.to_csv(RA/"tham_chieu.csv",index=False,encoding="utf-8-sig")
print("✓ Thiết kế SHA256:",design_sig,"· đã lưu bảng cặp và tham chiếu.")


In [ ]:
# Ô 4 · CHUẨN HÓA ÂM THẬT VÀ THAM CHIẾU — 16 kHz FLAC, KHÔNG SỬA INPUT
from math import gcd

def thanh_mono_16k(path):
    x,sr=sf.read(path,dtype="float32",always_2d=True)
    if not x.size or sr<=0:raise ValueError("Audio rỗng")
    x=x.mean(axis=1)
    if not np.isfinite(x).all():raise ValueError("Âm có NaN/Inf")
    if sr!=SR:
        g=gcd(int(sr),SR);x=resample_poly(x,SR//g,int(sr)//g).astype("float32")
    return x

def path_true(row):return RA/"audio"/"that"/row.speaker/(row.cau_id+".flac")
def path_fake(row,b):return RA/"audio"/b/row.speaker/(row.cau_id+".flac")
def path_ref(spk):return RA/"ref"/(spk+".wav")
def ghi_audio(p,x):
    p.parent.mkdir(parents=True,exist_ok=True);tmp=Path(str(p)+".tmp")
    sf.write(tmp,x,SR,format="WAV" if p.suffix.lower()==".wav" else "FLAC",subtype="PCM_16")
    z=sf.info(tmp)
    if z.samplerate!=SR or z.frames==0:raise ValueError("Lưu tệp lỗi")
    os.replace(tmp,p)
for r in refs.itertuples(index=False):
    p=path_ref(r.speaker)
    if not p.is_file():ghi_audio(p,thanh_mono_16k(r.ref_wav))
for r in design.itertuples(index=False):
    p=path_true(r)
    if not p.is_file():ghi_audio(p,thanh_mono_16k(r.wav_that))
print("✓ Đã chuẩn bị",len(design),"clip thật và",len(refs),"WAV tham chiếu")


In [ ]:
# Ô 5 · SINH CLIP GIẢ THEO CẶP — CHỈ NHỮNG CLIP CÒN THIẾU
# Kiểm tra sơ bộ không chứng minh câu được đọc đầy đủ; xem report + nghe lại.
from omnivoice import OmniVoice
from vieneu import Vieneu
from math import gcd

def chuan_hoa(a,sr):
    if isinstance(a,tuple) and len(a)==2 and isinstance(a[1],(int,float)):
        x,_=chuan_hoa(a[0],int(a[1]));return x,int(a[1])
    if isinstance(a,dict):
        for k in ("audio","wav","waveform","data","arr","y"):
            if k in a:return chuan_hoa(a[k],sr)
        raise ValueError("Kết quả không có audio")
    if hasattr(a,"detach"):a=a.detach().float().cpu().numpy()
    if not isinstance(a,(np.ndarray,list)):
        for k in ("audio","wav","waveform"):
            if hasattr(a,k):return chuan_hoa(getattr(a,k),sr)
    a=np.asarray(a,dtype=np.float32)
    if a.ndim>1:a=a.mean(axis=0) if a.shape[0]<=8 else a.mean(axis=1)
    return np.ascontiguousarray(a.reshape(-1)),int(sr)

def sang_16k(x,sr):
    if sr<=0 or x.size==0 or not np.isfinite(x).all():raise ValueError("Âm rỗng hoặc không hợp lệ")
    if sr!=SR:
        g=gcd(int(sr),SR);x=resample_poly(x,SR//g,int(sr)//g).astype("float32")
    return x

def kiem_nghi(x,do_dai_that):
    if x.size==0 or not np.isfinite(x).all():return False,"âm rỗng hoặc NaN"
    duration=len(x)/SR;rms=float(np.sqrt(np.mean(x*x)))
    if duration<max(.7,.35*do_dai_that):return False,f"ngắn {duration:.2f}s so với thật {do_dai_that:.2f}s"
    if rms<0.005:return False,f"âm rất nhỏ/rỗng RMS={rms:.5f}"
    return True,f"{duration:.2f}s, rms={rms:.4f}"

def kiem_tep(p,do_dai_that):
    try:
        x,sr=sf.read(p,dtype="float32")
        if sr!=SR:return False,"tần số không khớp"
        return kiem_nghi(x,do_dai_that)
    except Exception as e:return False,f"{type(e).__name__}: {str(e)[:100]}"

ft_hits=sorted(set(glob.glob("/kaggle/input/**/checkpoint-4000/config.json",recursive=True)))
# Ưu tiên checkpoint-4000 thuộc omdiom_full, nhưng không tự chọn khi vẫn mơ hồ.
ft_hits=[p for p in ft_hits if "omdiom_full" in p.lower()]
if len(ft_hits)!=1:raise RuntimeError(f"Cần duy nhất checkpoint-4000/omdiom_full trong Input; tìm thấy {ft_hits}")
FT=str(Path(ft_hits[0]).parent);print("✓ Checkpoint OmDiOm:",FT)
log_path=RA/"nhat_ky_sinh.json"
logs=json.loads(log_path.read_text(encoding="utf-8")) if log_path.is_file() else []
def luu_log():
    tmp=Path(str(log_path)+".tmp");tmp.write_text(json.dumps(logs,ensure_ascii=False,indent=2),encoding="utf-8");os.replace(tmp,log_path)

t0=time.time();stop=False
for b in BO:
    need=[]
    for r in design.itertuples(index=False):
        p=path_fake(r,b)
        if not p.is_file() or not kiem_tep(p,r.giay_that)[0]:need.append(r)
    print(f"\n▶ {b}: cần xử lý {len(need)}/{len(design)} clip",flush=True)
    if not need:continue
    try:
        if b=="vieneu":
            model=Vieneu();native_sr=int(getattr(model,"sample_rate",0) or 48000)
            sig=inspect.signature(model.infer);co_ref_text="ref_text" in sig.parameters
            print("  VieNeu sample rate:",native_sr,"· ref_text:",co_ref_text)
        else:
            source="k2-fsa/OmniVoice" if b=="omnivoice" else FT
            model=OmniVoice.from_pretrained(source,device_map="cuda",dtype=torch.float16)
            native_sr=int(getattr(model,"sampling_rate",0) or 24000)
            print("  nạp:",source,"· sample rate:",native_sr)
    except Exception as e:
        logs.append({"bo_sinh":b,"trang_thai":"loi_nap_model","loi":f"{type(e).__name__}: {str(e)[:300]}"});luu_log();print("⚠️ Không nạp được",b,e,flush=True);continue
    by={}
    for r in need:by.setdefault(r.speaker,[]).append(r)
    for spk,rows in sorted(by.items()):
        if time.time()-t0>GIO_TOI_DA*3600:stop=True;break
        prompt=None
        if b!="vieneu":
            try:
                with torch.inference_mode():prompt=model.create_voice_clone_prompt(ref_audio=str(path_ref(spk)),ref_text=rows[0].ref_text)
            except Exception as e:
                logs.append({"bo_sinh":b,"speaker":spk,"trang_thai":"loi_prompt","loi":f"{type(e).__name__}: {str(e)[:300]}"});luu_log();print("⚠️ prompt",spk,e,flush=True);continue
        for r in rows:
            if time.time()-t0>GIO_TOI_DA*3600:stop=True;break
            p=path_fake(r,b);done=False
            # Ghi mỗi lần thử, để resume dùng seed mới thay vì lặp seed cũ.
            old_n=sum(1 for z in logs if z.get("pair_id")==r.pair_id and z.get("bo_sinh")==b and "seed" in z)
            base=int(hashlib.sha256((b+"/"+r.pair_id).encode()).hexdigest()[:8],16)
            for attempt in range(LAN_THU):
                seed=(base+old_n+attempt+SEED)%(2**32-1)
                record={"bo_sinh":b,"speaker":spk,"pair_id":r.pair_id,"seed":seed,"lan":attempt+1}
                try:
                    torch.manual_seed(seed);torch.cuda.manual_seed_all(seed);np.random.seed(seed)
                    with torch.inference_mode():
                        if b=="vieneu":
                            kw={"text":r.text,"ref_audio":str(path_ref(spk))}
                            if co_ref_text:kw["ref_text"]=r.ref_text
                            out=model.infer(**kw)
                        else:out=model.generate(text=r.text,language="vi",voice_clone_prompt=prompt,num_step=NUM_STEP,postprocess_output=True)
                    x,sr=chuan_hoa(out,native_sr);x=sang_16k(x,sr)
                    ok,why=kiem_nghi(x,r.giay_that)
                    if not ok:raise ValueError(why)
                    ghi_audio(p,x)
                    ok,why=kiem_tep(p,r.giay_that)
                    if not ok:raise ValueError("Kiểm tra sau ghi: "+why)
                    record.update(trang_thai="da_sinh_can_nghe",giay=len(x)/SR,rms=float(np.sqrt(np.mean(x*x))))
                    done=True
                except Exception as e:record.update(trang_thai="loi",loi=f"{type(e).__name__}: {str(e)[:250]}")
                logs.append(record);luu_log()
                if done:break
            if not done:print("  ⚠️ chưa sinh được:",b,r.pair_id,flush=True)
        print(f"  {b} · {spk}: {len(rows)} clip cần xử lý · {(time.time()-t0)/60:.1f} phút",flush=True)
        if stop:break
    try:
        if hasattr(model,"close"):model.close()
        del model
        import gc;gc.collect();torch.cuda.empty_cache()
    except Exception as e:print("⚠️ dọn mô hình:",e)
    if stop:break
if stop:print("⏰ Chạm giới hạn giờ; vẫn lập báo cáo và đóng gói phần đã sinh.")
print("✓ Kết thúc vòng sinh; xem nhật ký và kiểm tra chất lượng bằng tai.")


In [ ]:
# Ô 6 · MANIFEST, CẶP ĐỦ, CÂN BẰNG TEST THEO CẶP
all_rows=[];incomplete=[];audio_qc=[]
for r in design.itertuples(index=False):
    pt=path_true(r)
    pths={b:path_fake(r,b) for b in BO}
    if not pt.is_file():incomplete.append({"pair_id":r.pair_id,"ly_do":"thiếu thật"});continue
    missing=[]
    for b,p in pths.items():
        ok,why=kiem_tep(p,r.giay_that) if p.is_file() else (False,"không có tệp")
        if not ok:missing.append(b+": "+why)
    if missing:
        incomplete.append({"pair_id":r.pair_id,"ly_do":"; ".join(missing)});continue
    for b,p,label in [("that",pt,0)]+[(b,pths[b],1) for b in BO]:
        z=sf.info(p)
        all_rows.append(dict(id_clip=b+"/"+r.speaker+"/"+r.cau_id,pair_id=r.pair_id,speaker=r.speaker,tap=r.tap,cau_id=r.cau_id,text=r.text,nguon=b,nhan=label,duong_dan_input=str(p),sr=z.samplerate,giay=z.frames/z.samplerate))
COLM=["id_clip","pair_id","speaker","tap","cau_id","text","nguon","nhan","duong_dan_input","sr","giay"]
M=pd.DataFrame(all_rows,columns=COLM);pd.DataFrame(incomplete,columns=["pair_id","ly_do"]).to_csv(RA/"cap_chua_du.csv",index=False,encoding="utf-8-sig")
M.to_csv(RA/"manifest_TA.csv",index=False,encoding="utf-8-sig")
print("Cặp đầy đủ:",M.pair_id.nunique(),"/",len(design),"· loại khỏi train/test:",len(incomplete),"· clip:",len(M))
if len(M):
    print(pd.crosstab([M.tap,M.nguon],M.nhan).to_string())
    print("Người train/test:",M[M.tap=="train"].speaker.nunique(),"/",M[M.tap=="test"].speaker.nunique())
    assert set(M[M.tap=="train"].speaker).isdisjoint(set(M[M.tap=="test"].speaker))
    assert set(M[M.tap=="train"].pair_id).isdisjoint(set(M[M.tap=="test"].pair_id))
    assert M.groupby("pair_id").size().eq(4).all()
    assert M.sr.eq(SR).all()
# Danh sách test cân bằng: với mỗi câu, giữ thật và 1 giả; rải đều 3 bộ sinh.
held=M[M.tap=="test"];ids=sorted(held.pair_id.unique());selected=[]
for i,key in enumerate(ids):
    z=held[held.pair_id==key];b=BO[i%len(BO)]
    selected.extend(z[(z.nguon=="that")|(z.nguon==b)].id_clip.tolist())
bal_ids=set(selected)
M["test_can_bang"]=M.id_clip.isin(bal_ids)
M.to_csv(RA/"manifest_TA.csv",index=False,encoding="utf-8-sig")
print("Test cân bằng ghép cặp:",len(bal_ids),"clip =",len(ids),"thật +",len(ids),"giả; bộ sinh chọn luân phiên.")
print("⚠️ Kiểm tra âm lượng/thời lượng không bảo đảm nói đủ chữ; phải nghe báo cáo.")


## Ô 7 · Mã trích đặc trưng nhúng sẵn
Mã dưới đây được ghi từ chính notebook; không cần tải thêm tệp.

In [ ]:
%%writefile /kaggle/working/ta_vivos/bo_trich_dac_trung.py
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly, stft
from scipy.fft import dct
SR_DICH=16000
# Ô 3 · MÃ TRÍCH ĐẶC TRƯNG — 16 kHz, CÙNG QUY TRÌNH CHO 3 BỘ SINH
# Tự triển khai STFT + thang mel + DCT bằng numpy/scipy; không cần TTS hay GPU.
def _mel_filters(sr=16000,nfft=512,nmels=40):
    # Thang mel HTK và dải tam giác; tính một lần mỗi tiến trình.
    hz=np.linspace(0,sr/2,nfft//2+1)
    mel=lambda h:2595*np.log10(1+h/700)
    inv=lambda m:700*(10**(m/2595)-1)
    edges=inv(np.linspace(mel(0),mel(sr/2),nmels+2))
    flt=np.zeros((nmels,len(hz)),dtype=np.float64)
    for k in range(nmels):
        left=(hz-edges[k])/(edges[k+1]-edges[k]+1e-12)
        right=(edges[k+2]-hz)/(edges[k+2]-edges[k+1]+1e-12)
        flt[k]=np.maximum(0,np.minimum(left,right))
    return flt
MEL=_mel_filters()

def dac_trung_mot(row):
    from math import gcd
    try:
        y,sr=sf.read(row["duong_dan_input"],dtype="float32",always_2d=True)
        if sr<=0 or not y.size:raise ValueError("âm rỗng hoặc sample rate không hợp lệ")
        if int(sr)!=int(row["sr"]):raise ValueError(f"sample rate thực {sr} khác manifest {row['sr']}")
        y=y.mean(axis=1).astype(np.float64)
        if not np.isfinite(y).all():raise ValueError("âm có NaN/Inf")
        if sr!=SR_DICH:
            g=gcd(int(sr),SR_DICH);y=resample_poly(y,SR_DICH//g,int(sr)//g)
        if len(y)<int(0.2*SR_DICH):raise ValueError("clip dưới 0,2s")
        raw_rms=float(np.sqrt(np.mean(y*y)))
        if raw_rms<1e-8:raise ValueError("âm hoàn toàn im lặng")
        # Chuẩn hóa RMS clip để giảm sự phụ thuộc vào âm lượng đầu ra.
        y=y/raw_rms
        _,_,Z=stft(y,fs=SR_DICH,window="hann",nperseg=400,noverlap=240,nfft=512,boundary="zeros",padded=True)
        power=np.abs(Z)**2+1e-20
        logmel=np.log(MEL@power+1e-12)
        m=dct(logmel,type=2,axis=0,norm="ortho")[:13]
        dm=np.gradient(m,axis=1) if m.shape[1]>1 else np.zeros_like(m)
        f={}
        for name,A in [("mfcc",m),("delta",dm)]:
            for i in range(13):
                f[f"F1_{name}{i:02d}_mean"]=float(np.mean(A[i]))
                f[f"F1_{name}{i:02d}_std"]=float(np.std(A[i]))
        freq=np.linspace(0,SR_DICH/2,power.shape[0]);total=power.sum(axis=0)+1e-20
        centroid=(freq[:,None]*power).sum(axis=0)/total
        bandwidth=np.sqrt(((freq[:,None]-centroid)**2*power).sum(axis=0)/total)
        roll=freq[(np.cumsum(power,axis=0)>=.95*total).argmax(axis=0)]
        flat=np.exp(np.log(power).mean(axis=0))/(power.mean(axis=0)+1e-20)
        for name,v in [("centroid_hz",centroid),("bandwidth_hz",bandwidth),("rolloff95_hz",roll),("flatness",flat)]:
            f[f"F2_{name}_mean"]=float(np.mean(v));f[f"F2_{name}_std"]=float(np.std(v))
        for a,b in [(0,500),(500,1000),(1000,2000),(2000,4000),(4000,8001)]:
            ratio=power[(freq>=a)&(freq<b)].sum(axis=0)/total
            f[f"F2_band_{a}_{b}_mean"]=float(np.mean(ratio))
            f[f"F2_band_{a}_{b}_std"]=float(np.std(ratio))
        n=400;hop=160
        yp=np.pad(y,(n//2,n//2));fr=np.lib.stride_tricks.sliding_window_view(yp,n)[::hop]
        er=np.sqrt(np.mean(fr*fr,axis=1));zc=np.mean((fr[:,:-1]*fr[:,1:])<0,axis=1)
        threshold=max(float(er.max())*0.1,1e-12)
        f.update({"F3_duration_s":float(len(y)/SR_DICH),"F3_silence_fraction":float(np.mean(er<threshold)),
                  "F3_frame_rms_std":float(np.std(er)),"F3_zcr_mean":float(np.mean(zc)),"F3_zcr_std":float(np.std(zc))})
        if not all(np.isfinite(v) for v in f.values()):raise ValueError("Đặc trưng có NaN/Inf")
        return {"id_clip":row["id_clip"],**f},None
    except Exception as e:
        return None,{"id_clip":row["id_clip"],"duong_dan":row["duong_dan_input"],"loi":f"{type(e).__name__}: {str(e)[:300]}"}
print("F1: MFCC + delta (mean/std); F2: phổ; F3: thời lượng và nhịp. Không trích F0.")


In [ ]:
# Ô 8 · TRÍCH ĐẶC TRƯNG VÀ HUẤN LUYỆN — NẾU CÓ ĐỦ DỮ LIỆU
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (accuracy_score,balanced_accuracy_score,precision_recall_fscore_support,
    confusion_matrix,roc_curve,roc_auc_score,classification_report)
results=[];cvrows=[];chi_tiet={};pred_table=[];train_ok=False;thong_bao=""
try:
    if len(M)==0:raise RuntimeError("Không có cặp đủ ba bộ sinh; cần xem cap_chua_du.csv và nhật ký sinh.")
    a=M[M.tap=="train"];z=M[M.tap=="test"]
    if a.speaker.nunique()<3 or z.speaker.nunique()<1 or set(a.nhan)!={0,1} or set(z.nhan)!={0,1}:
        raise RuntimeError("Không đủ người hoặc nhãn để train/test theo người.")
    spec=importlib.util.spec_from_file_location("ta_trich",RA/"bo_trich_dac_trung.py");module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
    inputs=M.to_dict("records");rows=[];errs=[]
    for start in range(0,len(inputs),128):
        got=Parallel(n_jobs=N_WORK,prefer="threads")(delayed(module.dac_trung_mot)(q) for q in inputs[start:start+128])
        rows.extend(x for x,e in got if x is not None);errs.extend(e for x,e in got if e is not None)
        print(f"  đặc trưng {min(start+128,len(inputs))}/{len(inputs)} · lỗi {len(errs)}",flush=True)
    if errs:
        pd.DataFrame(errs).to_csv(RA/"loi_dac_trung.csv",index=False,encoding="utf-8-sig")
        raise RuntimeError(f"{len(errs)} lỗi trích đặc trưng; không loại clip âm thầm.")
    F=pd.DataFrame(rows);F.to_csv(RA/"dac_trung_TA.csv",index=False,encoding="utf-8-sig")
    D=M.merge(F,on="id_clip",validate="one_to_one")
    columns=sorted(c for c in F if c.startswith(("F1_","F2_","F3_")))
    if len(columns)!=75 or not np.isfinite(D[columns].to_numpy(float)).all():raise ValueError("Thiếu 75 đặc trưng hoặc có NaN/Inf")
    tr=D[D.tap=="train"].copy();te=D[D.tap=="test"].copy();be=te[te.test_can_bang].copy()
    if len(be)!=2*te.pair_id.nunique() or be.nhan.sum()!=te.pair_id.nunique():raise ValueError("Thiết kế test cân bằng không hợp lệ")
    folds=list(GroupKFold(n_splits=3).split(tr,groups=tr.speaker))
    for ti,vi in folds:
        if set(tr.iloc[ti].speaker)&set(tr.iloc[vi].speaker):raise ValueError("Rò rỉ người nói trong CV")
    def new_model(name):
        if name=="SVM":return Pipeline([("scale",StandardScaler()),("model",SVC(C=1,kernel="rbf",class_weight="balanced"))])
        return RandomForestClassifier(n_estimators=150,min_samples_leaf=2,class_weight="balanced",n_jobs=N_WORK,random_state=SEED)
    for name in ("SVM","RandomForest"):
        X=tr[columns].to_numpy(dtype=float);y=tr.nhan.to_numpy()
        for k,(ti,vi) in enumerate(folds,1):
            model=new_model(name);model.fit(X[ti],y[ti]);pred=model.predict(X[vi]);ba=balanced_accuracy_score(y[vi],pred)
            pr,re,f1,_=precision_recall_fscore_support(y[vi],pred,labels=[0,1],average="macro",zero_division=0)
            cvrows.append(dict(mo_hinh=name,fold=k,balanced_accuracy=float(ba),macro_f1=float(f1),nguoi_train=tr.iloc[ti].speaker.nunique(),nguoi_val=tr.iloc[vi].speaker.nunique()))
            print(f"  CV {name} fold {k}: balanced accuracy={ba:.4f}, macro-F1={f1:.4f}",flush=True)
    CVD=pd.DataFrame(cvrows);CVD.to_csv(RA/"CV_theo_nguoi.csv",index=False,encoding="utf-8-sig")
    select=CVD.groupby("mo_hinh",as_index=False).balanced_accuracy.mean().sort_values(["balanced_accuracy","mo_hinh"],ascending=[False,True])
    selected=str(select.iloc[0].mo_hinh);select.to_csv(RA/"chon_mo_hinh_train.csv",index=False,encoding="utf-8-sig")
    model=new_model(selected);model.fit(tr[columns].to_numpy(float),tr.nhan)
    joblib.dump({"model":model,"feature_names":columns,"labels":{"0":"that","1":"gia"},"sr":SR,"selection":"3-fold GroupKFold theo speaker; balanced accuracy tren train"},RA/"phan_loai_TA.joblib")
    def score_raw(model,X):
        if hasattr(model,"decision_function"):return model.decision_function(X)
        return model.predict_proba(X)[:,list(model.classes_).index(1)]
    for kind,data in [("toan_bo_1_that_3_gia",te),("can_bang_1_that_1_gia",be)]:
        X=data[columns].to_numpy(float);yy=data.nhan.to_numpy();pred=model.predict(X);scr=score_raw(model,X)
        pr,re,f1,_=precision_recall_fscore_support(yy,pred,labels=[0,1],average="macro",zero_division=0)
        auc=roc_auc_score(yy,scr);fpr,tpr,threshold=roc_curve(yy,scr);delta=fpr-(1-tpr);k=int(np.argmin(abs(delta)));eer=float((fpr[k]+(1-tpr[k]))/2)
        r=dict(tap_test=kind,n=len(data),mo_hinh=selected,accuracy=float(accuracy_score(yy,pred)),balanced_accuracy=float(balanced_accuracy_score(yy,pred)),precision_macro=float(pr),recall_macro=float(re),macro_f1=float(f1),roc_auc=float(auc),eer_xap_xi=eer)
        results.append(r);chi_tiet[kind]={"cm":confusion_matrix(yy,pred,labels=[0,1]).tolist(),"report":classification_report(yy,pred,labels=[0,1],target_names=["that","gia"],output_dict=True,zero_division=0),"fpr":fpr.tolist(),"tpr":tpr.tolist()}
        pred_table.append(data[["id_clip","pair_id","speaker","tap","cau_id","nguon","nhan"]].assign(tap_test=kind,du_doan=pred,diem_gia=scr))
        print(f"✓ {kind}: balanced accuracy={r['balanced_accuracy']:.4f} · macro-F1={r['macro_f1']:.4f} · AUC={auc:.4f} · EER xấp xỉ={eer:.4f}")
    R=pd.DataFrame(results);R.to_csv(RA/"ket_qua_TA.csv",index=False,encoding="utf-8-sig")
    predictions=pd.concat(pred_table,ignore_index=True);predictions.to_csv(RA/"du_doan_test_TA.csv",index=False,encoding="utf-8-sig")
    fake=predictions[(predictions.tap_test=="toan_bo_1_that_3_gia")&(predictions.nhan==1)]
    pergen=fake.assign(dung=(fake.du_doan==1).astype(float)).groupby("nguon",as_index=False).agg(so_clip=("dung","size"),ti_le_phat_hien_gia=("dung","mean"))
    pergen.to_csv(RA/"do_nhay_tung_bo_sinh.csv",index=False,encoding="utf-8-sig")
    with open(RA/"chi_tiet_TA.json","w",encoding="utf-8") as f:json.dump(chi_tiet,f,ensure_ascii=False,indent=2)
    train_ok=True
    if (RA/"loi_huan_luyen.txt").exists():(RA/"loi_huan_luyen.txt").unlink()
    print("✅ Đã huấn luyện; chọn mô hình theo CV trên train:",selected)
except Exception as e:
    import traceback
    thong_bao=traceback.format_exc();(RA/"loi_huan_luyen.txt").write_text(thong_bao,encoding="utf-8")
    print("⚠️ Chưa thể huấn luyện:",type(e).__name__,e,"— vẫn tạo báo cáo/ZIP phần đã có.")


In [ ]:
# Ô 9 · HTML TỰ CHỨA: NGHE VÀ SO SÁNH + EDA + KẾT QUẢ T-A
import html,base64,io
plt.rcParams.update({"figure.facecolor":"white","axes.spines.top":False,"axes.spines.right":False,"font.size":10.5,"axes.titleweight":"bold","figure.autolayout":True})
figs=[];fdir=RA/"hinh";fdir.mkdir(exist_ok=True)
def _plot(title,note,fig,name):
    b=io.BytesIO();fig.savefig(b,format="png",dpi=145,bbox_inches="tight",facecolor="white");plt.close(fig);raw=b.getvalue();(fdir/(name+".png")).write_bytes(raw);figs.append((title,note,base64.b64encode(raw).decode()))
if len(M):
    ct=pd.crosstab(M.nguon,M.tap).reindex(index=["that"]+BO,columns=["train","test"],fill_value=0)
    fig,ax=plt.subplots(figsize=(8,3.8));ct.plot(kind="bar",ax=ax,rot=0,color=["#197f9a","#eea05b"],edgecolor="white");ax.set_ylabel("Số clip");ax.set_title("Tập theo nguồn và người nói");ax.legend(frameon=False)
    _plot("Số lượng theo nguồn", "Bộ VIVOS và bộ 7 prompt Bước 2 tách riêng. Chỉ các cặp đủ 3 bộ sinh mới được đưa vào phân loại.",fig,"01_nguon")
    fig,ax=plt.subplots(figsize=(9,3.4)); vals=[M[M.nguon==b].giay.to_numpy() for b in ["that"]+BO];parts=ax.boxplot(vals,patch_artist=True,showfliers=False)
    for p,c in zip(parts['boxes'],["#746ab0","#2689a1","#e68c56","#23a48c"]):p.set_facecolor(c);p.set_alpha(.8)
    ax.set_xticks(range(1,5),["that"]+BO);ax.set_ylabel("giây");ax.set_title("Thời lượng audio sau đồng nhất 16 kHz");ax.grid(axis="y",alpha=.2)
    _plot("Thời lượng", "Khác thời lượng có thể trở thành dấu hiệu phân loại; ngưỡng chất lượng có thể làm thay đổi phân bố. Không suy ra đọc đủ chữ chỉ từ thời lượng.",fig,"02_thoi_luong")
    ct=pd.crosstab(M.speaker,M.nguon).reindex(columns=["that"]+BO,fill_value=0);fig,ax=plt.subplots(figsize=(7,max(5,len(ct)*.23)));im=ax.imshow(ct.to_numpy(),aspect="auto",cmap="YlGnBu")
    ax.set_xticks(range(4),ct.columns,rotation=15);ax.set_yticks(range(len(ct)),ct.index,fontsize=7);ax.set_title("Số clip của mỗi người × nguồn");ax.grid(False)
    if ct.to_numpy().min()!=ct.to_numpy().max():fig.colorbar(im,ax=ax,shrink=.75,label="số clip")
    _plot("Độ đầy đủ theo người", "Mỗi hàng đếm clip của một người sau lọc cặp. Màu giống nhau không bảo đảm clip đọc đúng nội dung.",fig,"03_day_du")
if train_ok:
    fig,ax=plt.subplots(figsize=(6,4));cm=np.array(chi_tiet["can_bang_1_that_1_gia"]["cm"]);norm=np.divide(cm,cm.sum(axis=1,keepdims=True),out=np.zeros_like(cm,dtype=float),where=cm.sum(axis=1,keepdims=True)!=0);im=ax.imshow(norm,cmap="Blues",vmin=0,vmax=1)
    for i in range(2):
        for j in range(2):ax.text(j,i,f"{cm[i,j]}\n{norm[i,j]*100:.1f}%",ha="center",va="center",fontsize=13,color="white" if norm[i,j]>.55 else "#1d2937")
    ax.set_xticks([0,1],["thật","giả"]);ax.set_yticks([0,1],["thật","giả"]);ax.set_xlabel("Dự đoán");ax.set_ylabel("Nhãn thật");ax.set_title("Ma trận nhầm lẫn — test ghép cặp cân bằng");ax.grid(False);fig.colorbar(im,ax=ax,shrink=.75)
    _plot("Ma trận nhầm lẫn", "Mỗi cặp test chọn ngẫu nhiên tất định một trong ba bộ sinh theo vòng; chỉ là một phép đánh giá trên 7 người test dự kiến.",fig,"04_confusion")
    q=chi_tiet["toan_bo_1_that_3_gia"];fig,ax=plt.subplots(figsize=(6,4));ax.plot(q["fpr"],q["tpr"],lw=2.5,color="#197f9a");ax.plot([0,1],[0,1],"--",color="#999");ax.set_xlabel("False positive rate — thật bị gắn giả");ax.set_ylabel("True positive rate — giả được phát hiện");ax.set_title("ROC trên toàn bộ test 1 thật : 3 giả");ax.grid(alpha=.2)
    _plot("ROC", "AUC đánh giá xếp hạng điểm; EER được ước lượng tại ngưỡng giao trên test và không phải ngưỡng triển khai đã chọn trước.",fig,"05_roc")
    fig,ax=plt.subplots(figsize=(7,3.4));ax.bar(pergen.nguon,pergen.ti_le_phat_hien_gia,color=["#2689a1","#e68c56","#23a48c"]);ax.set_ylim(0,1.05);ax.set_ylabel("Tỉ lệ giả được phát hiện");ax.set_title("Độ nhạy theo bộ sinh — test")
    for i,v in enumerate(pergen.ti_le_phat_hien_gia):ax.text(i,v+.02,f"{v:.3f}",ha="center")
    _plot("Theo bộ sinh", "Đây là recall của lớp giả theo từng bộ sinh, không phải accuracy tổng thể và không chứng minh tổng quát trên bộ sinh mới.",fig,"06_bo_sinh")

# Mỗi mẫu nghe là một bản thật và các bản giả được ghép theo cùng người/câu.
def data_audio(p):
    with open(p,"rb") as f:b=base64.b64encode(f.read()).decode()
    return "data:audio/flac;base64,"+b
clips=[]
if len(M):
    # Ưu tiên một cặp từ mỗi người test đầu tiên; tối đa 3 cặp để HTML không quá lớn.
    ds=M[M.tap=="test"].drop_duplicates("pair_id").drop_duplicates("speaker").head(3)
    for r in ds.itertuples(index=False):
        block=["<div class='card'><h3>"+html.escape(r.speaker)+" · "+html.escape(r.cau_id)+"</h3><p>"+html.escape(r.text)+"</p><div class='grid'>"]
        for b in ["that"]+BO:
            rec=M[(M.pair_id==r.pair_id)&(M.nguon==b)]
            if rec.empty:block.append("<div class='item'>"+html.escape(b)+": thiếu</div>");continue
            x=rec.iloc[0]
            try:block.append("<div class='item'><b>"+html.escape(b)+"</b> · "+f"{x.giay:.2f}s"+"<audio controls preload='none' src='"+data_audio(x.duong_dan_input)+"'></audio></div>")
            except Exception as e:block.append("<div class='item'>Không đọc được âm: "+html.escape(str(e))+"</div>")
        block.append("</div></div>");clips.append("".join(block))
style="""body{margin:0;background:#f3f7fa;color:#142d3a;font:16px/1.55 system-ui,-apple-system,Segoe UI,sans-serif}header{background:linear-gradient(130deg,#123446,#1c8094);color:white;padding:40px 22px}header div,main{max-width:1100px;margin:auto}main{padding:22px 20px 70px}h1{margin:0}h2{color:#17667b;margin-top:42px;border-bottom:2px solid #dae7ec}figure,.card{background:white;border:1px solid #dce7ed;border-radius:12px;padding:18px;margin:18px 0;overflow:auto}figure img{max-width:100%;display:block;margin:auto}figcaption{color:#526771;border-left:3px solid #2c97ac;padding-left:12px;font-size:14px}.warn{background:#fff7e8;border-left:4px solid #efad39;padding:16px}.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:12px}.item{border:1px solid #dce6ed;border-radius:9px;padding:12px}audio{width:100%;display:block}.table{border-collapse:collapse;width:100%;font-size:13px;white-space:nowrap}.table td,.table th{border-bottom:1px solid #e4eaf0;padding:8px}.table th{background:#e7f2f5;text-align:left}@media print{figure{break-inside:avoid}}"""
P=["<!doctype html><html lang='vi'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'><title>T-A VIVOS thật giả</title><style>",style,"</style></head><body><header><div><h1>T-A · Phân loại giọng thật / giả</h1><p>VIVOS · 25 người dự kiến · train/test tách theo người · 3 bộ sinh</p></div></header><main>"]
P.append("<div class='warn'><b>Giới hạn:</b> Chỉ cặp có đủ ba bộ sinh và qua ngưỡng sơ bộ được phân tích; điều này có thể tạo thiên lệch lựa chọn. Không có bảo đảm clip giả đọc đủ chữ. Người test chưa xuất hiện trong train của bộ phân loại, nhưng chưa biết họ có xuất hiện trong dữ liệu huấn luyện TTS nền hay không. Giấy phép VIVOS cần tuân thủ CC BY-NC-SA 4.0.</div>")
P.append(f"<h2>Tình trạng dữ liệu</h2><div class='card'><p>{M.pair_id.nunique()} / {len(design)} cặp đủ · {len(incomplete)} cặp chưa đủ · {len(M)} clip được phân tích. Mục tiêu 375 cặp / 1.500 clip. Train/test tách theo người, không đưa bộ 7 prompt vào.</p></div>")
P.append("<h2>Nghe cùng câu · cùng người</h2><p>So bản thật với ba bản giả, nghe rõ chữ, ngắt nghỉ và âm lượng. Tên người là mã VIVOS; không khẳng định danh tính clone chỉ dựa vào ngưỡng số.</p>"+("".join(clips) if clips else "<div class='warn'>Chưa có cặp test đủ để nghe.</div>"))
P.append("<h2>Chọn mô hình và kết quả</h2>")
if train_ok:
    P.append("<div class='card'><p>Chọn mô hình theo balanced accuracy trong 3-fold GroupKFold trên 18 người train; không chọn theo test. Test đầy đủ có 1 thật : 3 giả; test cân bằng chọn 1 giả cho mỗi bản thật theo vòng ba bộ sinh. EER là xấp xỉ từ ROC test, không phải ngưỡng vận hành đã khóa trước.</p>"+pd.DataFrame(results).to_html(index=False,classes='table',border=0,float_format=lambda x:f"{x:.4f}")+"</div>")
else:P.append("<div class='warn'>Chưa có kết quả phân loại: "+html.escape(thong_bao[-1000:])+"</div>")
P.append("<h2>Biểu đồ EDA và đánh giá</h2>")
for title,note,img in figs:P.append("<figure><h3>"+html.escape(title)+"</h3><img src='data:image/png;base64,"+img+"'><figcaption>"+html.escape(note)+"</figcaption></figure>")
P.append("<h2>Thiết kế và dữ liệu</h2><div class='card'><p>Áp dụng đồng nhất 16 kHz / mono / PCM16 FLAC, sau đó trích 75 đặc trưng có chuẩn hóa RMS theo clip. Bản thật có một tệp, bản giả ba tệp: class_weight cân bằng trong train; báo cả kết quả đầy đủ và test cân bằng. Dữ liệu lọc chất lượng có thể gây thiên lệch; benchmark này không thể khẳng định phát hiện mọi dạng giọng giả.</p><p>Nguồn VIVOS: CC BY-NC-SA 4.0; dữ liệu và trọng số có giấy phép riêng cần tuân thủ.</p></div></main></body></html>")
report=RA/"BAO_CAO_TA.html";report.write_text("".join(P),encoding="utf-8")
print(f"✓ HTML {report} · {len(figs)} hình · {len(clips)} cặp nghe · train_ok={train_ok}")


In [ ]:
# Ô 9b · LƯU HƯỚNG DẪN VÀ CHƯƠNG TRÌNH DỰ ĐOÁN
(RA/'du_doan_audio_TA.py').write_text("import sys, json, importlib.util\nfrom pathlib import Path\nimport numpy as np, soundfile as sf, joblib\nroot=Path(__file__).resolve().parent\nif len(sys.argv)!=2:\n    raise SystemExit('Cách dùng: python du_doan_audio_TA.py duong_dan_audio.wav')\naudio=Path(sys.argv[1])\nif not audio.is_file():raise SystemExit('Không tìm thấy audio: '+str(audio))\nmodel_path=root/'phan_loai_TA.joblib'\nif not model_path.is_file():raise SystemExit('Chưa có mô hình: kiểm tra kết quả huấn luyện và lỗi trong báo cáo.')\nspec=importlib.util.spec_from_file_location('extractor_TA',root/'bo_trich_dac_trung.py')\nm=importlib.util.module_from_spec(spec);spec.loader.exec_module(m)\nsr=sf.info(audio).samplerate\nfeature,error=m.dac_trung_mot({'id_clip':'audio_moi','duong_dan_input':str(audio),'sr':sr})\nif error:raise SystemExit('Không trích được đặc trưng: '+str(error))\nobj=joblib.load(model_path);X=np.array([[feature[n] for n in obj['feature_names']]],dtype=float)\npred=int(obj['model'].predict(X)[0])\nprint(json.dumps({'duong_dan':str(audio),'ket_qua':'gia' if pred else 'that', 'luu_y':'Chi la du doan tham khao tren mien VIVOS va ba bo sinh da nghien cuu.'},ensure_ascii=False,indent=2))\n",encoding='utf-8')
(RA/'HUONG_DAN_TA.md').write_text('# Hướng dẫn sử dụng kết quả T-A\n\n- `BAO_CAO_TA.html`: mở bằng trình duyệt, biểu đồ và clip nghe được nhúng sẵn.\n- `manifest_TA.csv`, `thiet_ke_cap.csv`: dữ liệu ghép cặp và thiết kế. Đường dẫn tuyệt đối trong CSV là vị trí khi chạy Kaggle, không còn đúng trên máy khác; thư mục `audio/` trong ZIP chứa các tệp theo nguồn/người/mã câu.\n- `CV_theo_nguoi.csv`: chọn mô hình theo ba fold chia người trên train. `ket_qua_TA.csv`, `du_doan_test_TA.csv`: chỉ có khi huấn luyện thành công. `cap_chua_du.csv`, `nhat_ky_sinh.json`: cặp và lần sinh chưa thành công; các ngưỡng không chứng minh nội dung đúng.\n- `phan_loai_TA.joblib`: mô hình nếu huấn luyện thành công. `bo_trich_dac_trung.py`, `du_doan_audio_TA.py`: mã dự đoán.\n\n## Dự đoán âm thanh mới\nGiải nén ZIP, giữ các tệp trên cùng thư mục. Cài Python cùng các gói `numpy`, `scipy`, `soundfile`, `scikit-learn`, `joblib` tương thích phiên bản ghi trong `thong_tin_TA.json`. Trên PowerShell tại thư mục đã giải nén chạy:\n\n```powershell\npython du_doan_audio_TA.py "D:\\duong_dan\\audio.wav"\n```\n\nKhông cần TTS hoặc GPU để phân loại. Mô hình chỉ là thử nghiệm trong miền VIVOS và ba bộ sinh được khảo sát; không dùng kết quả làm bằng chứng xác thực giọng nói. Chỉ mở tệp joblib đáng tin cậy vì việc giải tuần tự có thể chạy mã.\n',encoding='utf-8')
print('✓ Đã lưu chương trình và hướng dẫn dự đoán')


In [ ]:
# Ô 10 · THÔNG TIN & ĐÓNG GÓI MỘT ZIP, KỂ CẢ NẾU CÒN CẶP LỖI
info={"nguon":str(VIVOS),"thiet_ke_sha256":design_sig,"seed":SEED,"num_step":NUM_STEP,"so_nguoi_du_kien":SO_NGUOI,"so_cap_du_kien":len(design),"so_cap_du":int(M.pair_id.nunique()) if len(M) else 0,"so_cap_chua_du":len(incomplete),"da_huan_luyen":train_ok,"tieu_chi_sang_loc":">= max(0.7s, 0.35 x giay that), rms >=0.005; chi la heuristic, nghe kiem tra", "so_test_can_bang":len(bal_ids),"python":sys.version,"sklearn":sklearn.__version__,"numpy":np.__version__,"checkpoint":FT,"canh_bao":"test nguoi noi moi doi voi classifier, khong biet TTS pretraining; selection bias; khong co T-B/T-C"}
(RA/"thong_tin_TA.json").write_text(json.dumps(info,ensure_ascii=False,indent=2),encoding="utf-8")
name="/kaggle/working/KET_QUA_TA_VIVOS_"+time.strftime("%m%d_%H%M%S")
z=shutil.make_archive(name,"zip",RA)
print(f"\n📦 {z} · {os.path.getsize(z)/1024**2:.1f} MiB")
print("📄",report,"· mô hình:","có" if train_ok else "chưa")
print("Gửi BAO_CAO_TA.html, manifest_TA.csv, ket_qua_TA.csv (nếu có), cap_chua_du.csv và nhat_ky_sinh.json.")
print("Nếu cắt phiên, Save Version / lưu output, Add Input output ta_vivos cũ để notebook phục hồi theo vân tay thiết kế.")
